# Calculating common activity and light metrics

After preprocessing actigraphy data using the methods defined in the `Raw` class, users can begin calculating metrics based on the activity and light time series. These include common circadian rhythm metrics such as Interdaily Stability (IS) and Relative Amplitude (RA), among others.

<div class="alert alert-block alert-info">
<b>Contrast with pyActigraphy</b><p>A key architectural difference between <code>circStudio</code> and <code>pyActigraphy</code> lies in how metric functions are implemented. In <code>pyActigraphy</code>, metrics are defined as methods of the <code>BaseRaw</code> class, which inherits from the mixin subclasses <code>MetricsMixin</code>, <code>ScoringMixin</code> and <code>SleepBoutMixin</code>, and initializes a <code>LightRecording</code> instance to store the light intensity data.</p>

<p>In contrast, <code>circStudio</code> separates data structure from metric computation. The <code>Raw</code> class contains only the data attributes of the actigraphy recording and methods for transforming them (e.g., masking, binarization, imputation).</p>

<p>Metric computation functions in <code>circStudio</code> are not bound to a class. Instead, they are standalone functions that accept input data directly. This design gives users the flexibility to either use the <code>Raw</code> class for preprocessing or bypass it entirely and use their own custom-preprocessed data to compute metrics using <code>circStudio</code>'s core functionality.</p>
</div>

As before, we will import `circStudio` and `os`:

In [1]:
import circStudio
from circStudio.analysis import *
import os

Open the sample RPX file containing the data that will be used in this tutorial:

In [2]:
# Create file path for sample files within the circStudio package
fpath = os.path.join(os.path.dirname(circStudio.__file__))

# Create a new Raw instance using awd RPX adaptor
raw = circStudio.io.read_rpx(os.path.join(fpath, 'data', 'test_sample_rpx_eng.csv'), 
                             start_time='2015-07-04 12:00:00',
                             light_mode='White Light',
                             period='6 days',
                             language='ENG_UK')

In the next code blocks, we demonstrate how to compute several circadian rhythm metrics. These represent just a small subset of the available metrics in `circStudio`. Users can also implement their own custom metrics by following a similar function design.

### Activity

#### Average daily activity (ADAT)

In [ ]:
adat(data=raw.activity)

#### Interdaily stability (IS)

In [4]:
IS(data=raw.activity)

np.float64(0.469182266891273)

#### Interdaily variability (IV)

In [ ]:
IV(data=raw.activity)

#### Ten most active hours of the day (M10)

Returns a tuple with the onset and value of the M10.

In [ ]:
m10(data=raw.activity)

#### Five least active hours of the day (L5)

Returns a tuple with the onset and value of the L5.

In [ ]:
l5(data=raw.activity)

### Light

#### Average daily light profile

By default, `daily_profile` function returns a `pd.Series` containing the average daily light intensity values. However, its behavior can be modified to generate an interactive daily profile plot instead. For example, instead of:

In [ ]:
daily_profile(data=raw.light)

Simply do:

In [ ]:
daily_profile(data=raw.light, plot=True, log=True)

#### Interdaily stability (IS)

In [ ]:
IS(data=raw.light)

#### Interdaily variability (IV)

In [ ]:
IV(data=raw.activity)

#### Ten brightest hours of the day (M10)

In [ ]:
m10(data=raw.light)

#### Five least illuminated hours of the day (L5)

In [ ]:
l5(data=raw.light)

### Sleep

#### Automatic inactivity detection algorithms: Crespo and Roenneberg

`circStudio` contains several algorithms to detect inactivity periods in actigraphy recordings. In this tutorial, we will illustrate this functionality using the Crespo and Roenneberg (also known as MASDA) algorithms. You can use the 'plot' parameter to determine whether a `pd.Series` or an interactive plot must be returned.

If you set `plot` to `True`:

In [ ]:
Crespo(data=raw.activity, frequency=raw.frequency, plot=True)

By default, `plot` is set to `False`, thus returning a `pd.Series`:

In [ ]:
Crespo(data=raw.activity, frequency=raw.frequency)

Using now the Roenneberg algorithm:

In [ ]:
Roenneberg(data=raw.activity, plot=True)

#### Sleep transition probabilities

In [3]:
# Create file path for sample files within the circStudio package
fpath = os.path.join(os.path.dirname(circStudio.__file__))
raw = circStudio.io.read_awd(os.path.join(fpath, 'data', 'example_01.AWD'), period='7 days')

In [4]:
raw.apply_filters(threshold=0)

In [5]:
kRA(data=raw.activity)

np.float64(0.12337472433074494)

#### WASO

`circStudio` implements a method to approximate the WASO (wake after sleep onset metric) based on the use of the Roenneberg algorithm (which detects consolitated periods of sleep) and a epoch-by-epoch rest/activity scoring algorithm (either Cole-Kripke, Sadeh or Scripps) to detect wake periods during a consolidated period of sleep. The functionality returns a tuple containing daily waso, as well as mean waso:

In [ ]:
waso(data=raw.activity, frequency=raw.frequency, algo='Cole-Kripke', settings='mean') # WASO computed using the Sadeh Cole-Kripke algorithm

In [ ]:
waso(data=raw.activity, frequency=raw.frequency, algo='Sadeh') # WASO computed using the Sadeh algorithm

In [ ]:
waso(data=raw.activity, frequency=raw.frequency, algo='Scripps', settings='mean') # WASO computed using the Scripps algorithm

#### Sleep diaries

##### Introduction and opening sample files

`circStudio` keeps the ability to read sleep diaries in the form:

| SubjectID        | your_subject_id  |                  |
|------------------|------------------|------------------|
| Type             | Start            | End              |
| Night/Nap/NoWear | YYYY-MM-DD HH:MM | YYYY-MM-DD HH:MM |

Similar to `pyActigraphy`, it is possible to compute summary statistics and visualize the information contained in the sleep diary. In `circStudio`, functions to compute sleep efficiency (`sleep_efficiency`) and sleep onset latency (`sleep_onset_latency`) were implemented. These functions require both a sleep diary and an actigraphy recording containing an activity time series.

To illustrate, load a sample AWD file and the respective sleep diary:

In [ ]:
fpath = os.path.join(os.path.dirname(circStudio.__file__), 'data/')

# Load actigraphy file
raw = circStudio.io.read_awd(fpath+'example_01.AWD')

# Load diary
diary = raw.read_sleep_diary(fpath + 'example_01_sleepdiary.ods')

##### Accessing the dataframe and obtaining summary statistics

You can access the sleep_diary either by calling the object or by printing it:

In [ ]:
# Show sleep diary as a dataframe
raw.sleep_diary()

In [ ]:
# Print sleep diary as str
print(raw.sleep_diary)

Obtain summary statistics:

In [ ]:
raw.sleep_diary.summary()

##### Sleep efficiency and sleep onset latency from sleep diary and actigraphy

`circStudio` provides functions that integrate data from both the sleep diary and actigraphy recording to estimate <b>sleep efficiency</b> and <b>sleep onset latency</b>. For these functions to return a meaningful result, it is essential that the algorithmically detected sleep onset occurs after the bedtime recorded in the diary. Therefore, participants must be instructed to accurately report the time they go to bed. 

Sleep efficiency is computed as the ratio of average <b>total sleep time</b> (as estimated from actigraphy using the Roenneberg algorithm) to average <b>total time in bed</b> (as recorded in the diary):

In [ ]:
raw.sleep_diary.sleep_efficiency(data=raw.activity)

<b>Sleep onset latency</b> is calculated as the time difference between the <b>diary-reported bedtime</b> and the algorithmically detected <b>sleep onset</b> (Roenneberg algorithm). Days in which the recorded bedtime occurs <i>after</i>, likely due to diary entry errors, the detected sleep onset are excluded from the analysis.

In [ ]:
raw.sleep_diary.sleep_onset_latency(data=raw.activity)

As part of the open development philosophy of `circStudio`, feedback and suggestions for improving these functions are welcome!

##### Visualizing sleep diaries

An interactive plot of the sleep diary can be easily generated using the `plot()` method:

In [ ]:
raw.sleep_diary.plot(data=raw.activity)

Different sleep diary states have a distinct color associated with them, which could be changed:

In [ ]:
raw.sleep_diary.state_colour

For example, suppose that you want to change the color associated with the 'NIGHT' from light grey to a dark purple. You can run the command:

In [ ]:
# Change state color for 'NIGHT' events
raw.sleep_diary.state_colour['NIGHT'] = 'rgb(140,95,148)'

# Draw a new plot
raw.sleep_diary.plot(data=raw.activity)

In the following tutorial, we will explore the application of mathematical models of circadian rhythms to actigraphy data analysis.